In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = spark.table("launch_reliability.silver.launch_joined")

# --- windows ---
order = ["launch_jd", "launch_tag"]
earlier = (Window.unboundedPreceding, -1)   # every launch before this one

vehicle       = Window.partitionBy("lv_type").orderBy(*order)
vehicle_prior = vehicle.rowsBetween(*earlier)
family        = Window.partitionBy("lv_family").orderBy(*order)
family_prior  = family.rowsBetween(*earlier)
agency        = Window.partitionBy("agency").orderBy(*order)

failed = F.col("failed").cast("int")
has_family = F.col("lv_family").isNotNull()

# --- step 1: experience ---
gold = (df
    .withColumn("vehicle_flight_num", F.row_number().over(vehicle))
    .withColumn("vehicle_prior_failures",
                F.coalesce(F.sum(failed).over(vehicle_prior), F.lit(0)))
    .withColumn("days_since_prev_vehicle_launch",
                F.col("launch_jd") - F.lag("launch_jd").over(vehicle))
    .withColumn("agency_flight_num", F.row_number().over(agency))
    .withColumn("family_flight_num",
                F.when(has_family, F.row_number().over(family)))
    .withColumn("family_prior_failures",
                F.when(has_family, F.coalesce(F.sum(failed).over(family_prior), F.lit(0))))
)

# --- step 2: flight number of this rocket's last earlier failure ---
gold = gold.withColumn("last_failure_flight_num",
    F.max(F.when(F.col("failed"), F.col("vehicle_flight_num"))).over(vehicle_prior))

# --- step 3: recovery ---
gold = (gold
    .withColumn("flights_since_last_failure",
                F.col("vehicle_flight_num") - F.col("last_failure_flight_num") - 1)
    .withColumn("return_to_flight",
                F.coalesce(F.col("flights_since_last_failure") == 0, F.lit(False)))
)

# --- save ---
(gold.write.mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable("launch_reliability.gold.launch_features"))

print(f"gold.launch_features: {gold.count():,} rows")

Builds experience features for each launch using only launches that came before it.

- Vehicle experience: flight number, prior failures, and days since the rocket’s previous launch
- Family experience: flight number and prior failures across all variants of the rocket family
- Provider experience: the provider’s launch count
- Recovery: flights since the rocket’s last failure, and a return-to-flight flag for the first launch after a failure
Validated against Falcon 1’s history (three failures, then success on flight 4)

Output: gold.launch_features, one row per launch, ready for EDA and modeling.